In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
sc.settings.set_figure_params(dpi=300)

In [ ]:
spatial_adata = sc.read_10x_h5('../data/dumitru/xenium/cell_feature_matrix.h5')
spatial_adata

In [ ]:
cell_meta = pd.read_parquet('../data/dumitru/xenium/cells.parquet')
cell_meta['cell_id'] = cell_meta['cell_id'].apply(lambda x: x.decode('utf-8')) # convert byte strings to normal strings
cell_meta.set_index("cell_id", inplace=True) # set index to cell id
cell_meta

In [ ]:
spatial_adata.obs = spatial_adata.obs.join(cell_meta) # join metadata to obs
spatial_adata.obs

In [ ]:
spatial_adata.obsm["spatial"] = spatial_adata.obs[["x_centroid", "y_centroid"]].values # set spatial

In [ ]:
sc.pl.embedding(spatial_adata, basis='spatial', color=['total_counts', 'cell_area', 'nucleus_area'], size=3)

In [ ]:
# Saving count data
spatial_adata.layers["counts"] = spatial_adata.X.copy()

# Normalize and log-transform the data 
sc.pp.normalize_total(spatial_adata, target_sum=1e4) 
sc.pp.log1p(spatial_adata)

In [ ]:
sc.tl.pca(spatial_adata)
sc.pp.neighbors(spatial_adata)
sc.tl.umap(spatial_adata)

In [ ]:
sc.pl.umap(spatial_adata, color=['total_counts', 'cell_area', 'nucleus_area'])

In [ ]:
sc.tl.leiden(spatial_adata, resolution=0.3)

In [ ]:
sc.pl.umap(spatial_adata, color=['leiden'], legend_loc='on data')

In [ ]:
sc.pl.umap(spatial_adata, color=['SOX2', 'HOPX', 'EGFR', 'ASCL1'], ncols=2)

In [ ]:
sc.pl.embedding(spatial_adata, basis='spatial', color=['SOX2', 'HOPX', 'EGFR', 'ASCL1', 'HES5', 'NES', 'PROX1', 'ASCL1', 'EGFR', 'EOMES', 'MKI67', 'DCX'], layer='counts', size=4, vmax='p99')

In [ ]:
sc.pl.embedding(spatial_adata, basis='spatial', color=['leiden'], size=3)

In [ ]:
# Define NSCs based on marker expression
nsc_pos_markers = ['SOX2', 'NES', 'ASCL1', 'PAX6']
nsc_neg_markers = ['SOX10', 'OLIG2']
spatial_adata.obs['dumitru_nsc'] = ((spatial_adata[:, nsc_pos_markers].layers['counts'] > 0).sum(axis=1) == len(nsc_pos_markers)) & \
                            ((spatial_adata[:, nsc_neg_markers].layers['counts'] > 0).sum(axis=1) == 0)

# Plot NSCs spatially
import matplotlib.pyplot as plt

coords = spatial_adata.obsm['spatial']
mask_true = spatial_adata.obs['dumitru_nsc'] == True
mask_false = spatial_adata.obs['dumitru_nsc'] == False

plt.figure(figsize=(8, 8))
plt.scatter(coords[mask_false, 0], coords[mask_false, 1], c='lightgray', s=2, label='False')
plt.scatter(coords[mask_true, 0], coords[mask_true, 1], c='red', s=10, label='True', alpha=0.8)
plt.axis('off')
plt.legend(frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
spatial_adata.obs['dumitru_nsc'].value_counts()

In [ ]:
nsc_adata = spatial_adata[spatial_adata.obs['dumitru_nsc']].copy()
nsc_adata

In [ ]:
sc.tl.pca(nsc_adata)
sc.pp.neighbors(nsc_adata)
sc.tl.umap(nsc_adata)

In [ ]:
sc.tl.leiden(nsc_adata, resolution=0.4)

In [ ]:
# Define custom colors for leiden clusters
custom_colors = ['blue', 'green', 'purple', 'orange']
nsc_adata.uns['leiden_colors'] = custom_colors

In [ ]:
sc.pl.umap(nsc_adata, color=['leiden'])

In [ ]:
sc.tl.rank_genes_groups(nsc_adata, groupby='leiden')
sc.pl.rank_genes_groups(nsc_adata, n_genes=20)

In [ ]:
import matplotlib.pyplot as plt

# Create figure
fig, ax = plt.subplots(figsize=(10, 10))

# Plot background in gray
ax.scatter(spatial_adata.obsm['spatial'][:, 0], 
           spatial_adata.obsm['spatial'][:, 1], 
           c='lightgray', s=3, alpha=0.8, edgecolors='none')

# Plot clusters 0, 1, 2 first
for cluster in ['1', '3']:
    mask = nsc_adata.obs['leiden'] == cluster
    sc.pl.embedding(nsc_adata[mask], basis='spatial', color='leiden', 
                    size=180, ax=ax, show=False)

ax.set_aspect('equal')
ax.axis('off')
plt.tight_layout()

In [ ]:
adata = sc.read_h5ad('../data/dumitru/single_nucleus/h5ad/dumitru_all_donors_nsc_annotations.h5ad')
adata

In [ ]:
# calculate average gene expression per group
adata_avg = adata.to_df().groupby(adata.obs['cell_type']).mean()
adata_avg

In [ ]:
# common genes
common_genes = adata_avg.columns.intersection(spatial_adata.var_names)
common_genes

In [ ]:
from scipy.sparse import issparse

def compute_spatial_vs_celltype_corr(spatial_adata, adata_avg):
    # Align genes
    genes = spatial_adata.var_names.intersection(adata_avg.columns)
    X = spatial_adata[:, genes].X
    X = X.toarray() if issparse(X) else X
    Y = adata_avg[genes].values

    # Row-wise center (across genes) and compute Pearson correlations
    eps = 1e-8
    Xc = X - X.mean(axis=1, keepdims=True)
    Yc = Y - Y.mean(axis=1, keepdims=True)
    denom = np.linalg.norm(Xc, axis=1, keepdims=True) * np.linalg.norm(Yc, axis=1)
    corr = (Xc @ Yc.T) / (denom + eps)

    # Assign results
    corr_df = pd.DataFrame(corr, index=spatial_adata.obs_names, columns=adata_avg.index)
    for col in corr_df.columns:
        spatial_adata.obs[f"corr_{col}"] = corr_df[col].values
    spatial_adata.obs["pred_cell_type"] = corr_df.idxmax(axis=1).values
    return corr_df

corr_df = compute_spatial_vs_celltype_corr(spatial_adata, adata_avg)

In [ ]:
sc.pl.embedding(spatial_adata, basis='spatial', color=['corr_Astrocyte', 'corr_nsc_01', 'corr_nsc_02', 'corr_nsc_03', 'corr_nsc_04', 'corr_nsc_05'], size=3, ncols=3, cmap='coolwarm', vmax=0.5)

In [ ]:
sc.pl.embedding(spatial_adata, basis='spatial', color=['pred_cell_type'], size=3)

In [ ]:
xmin, xmax = 1000, 3000 
ymin, ymax = 5200, 7700

# Build mask using the stored centroids
x = spatial_adata.obsm["spatial"][:, 0]
y = spatial_adata.obsm["spatial"][:, 1]
mask = (x >= xmin) & (x <= xmax) & (y >= ymin) & (y <= ymax)

# Subset to zoomed region
zoom_adata = spatial_adata[mask].copy()
zoom_adata

In [ ]:
sc.pl.embedding(zoom_adata, basis="spatial", color=["pred_cell_type"], size=15)

In [ ]:
sc.pl.embedding(zoom_adata, basis='spatial', color=['corr_nsc_01', 'corr_nsc_02', 'corr_nsc_03', 'corr_nsc_04', 'corr_nsc_05'], size=20, ncols=3, cmap='coolwarm', vmax=0.4)

In [ ]:
xmin, xmax = 3000, 6000 
ymin, ymax = 5000, 8000

# Build mask using the stored centroids
x = spatial_adata.obsm["spatial"][:, 0]
y = spatial_adata.obsm["spatial"][:, 1]
mask = (x >= xmin) & (x <= xmax) & (y >= ymin) & (y <= ymax)

# Subset to zoomed region
zoom_adata = spatial_adata[mask].copy()
zoom_adata

In [ ]:
sc.pl.embedding(zoom_adata, basis='spatial', color=['ASCL1', 'HOPX'], size=20, cmap='coolwarm', frameon=False)